# BioSkin Spectral Reflectance Note

This notebook documents a compact BioSkin workflow for mapping biophysical skin parameters to visible and near-infrared spectral reflectance. The emphasis is on the forward decoder provided by BioSkin and on clear visualization of the decoded reflectance curve.

## Methodological Overview


### Forward Process

The full forward process starts from biophysical skin parameters, estimates spectral reflectance with a skin optics model, and renders the spectrum into a color representation.

```mermaid
flowchart LR
    A[Biophysical skin parameters] --> B[Human skin model]
    B --> C[Spectral reflectance]
    C --> D[Render to RGB]
    D --> E[Color system<br/>RGB / CIE LAB]
```

#### Notes

- **Skin parameters:** melanin volume fraction, blood volume fraction, layer thickness, hemoglobin oxygenation, melanin type ratio, and occlusion.
- **Human skin model:** brute-force Monte Carlo random walks inside the skin.
- **Output spectrum:** reflectance as a function of wavelength.

### Inverse Process

The inverse process maps an observed color or reflectance representation back to latent skin parameters. BioSkin implements this as an encoder-decoder cycle in which the encoder estimates parameters from RGB and the decoder reconstructs the corresponding spectrum.

```mermaid
flowchart LR
    A[Biophysical skin parameters] -->|Decoder| B[Spectral reflectance]
    B -->|Render| C[RGB]
    C -->|Encoder| A
```

#### Notes

- **Decoder:** maps skin parameters to spectral reflectance.
- **Renderer:** converts spectral reflectance into RGB.
- **Encoder:** maps RGB back to estimated skin parameters.
- This forms an inverse modeling framework for estimating biological skin parameters from color observations.

### Compact Summary

```text
Forward:
Biophysical skin parameters
    -> human skin model / brute-force Monte Carlo
    -> spectral reflectance
    -> RGB / CIE LAB

Inverse:
RGB
    -> encoder
    -> biophysical skin parameters
    -> decoder
    -> spectral reflectance
    -> render
    -> RGB
```


## Decoder Demonstration

The following cell records the BioSkin parameter names and their reported ranges. The regular range excludes localized effects such as rashes or spots, while the plausible and full ranges provide broader limits for parameter exploration.

In [15]:
import pandas as pd

SKIN_PROPS = [
    "Melanin",
    "Hemoglobin",
    "Thickness",
    "Blood oxygenation",
    "Melanin type ratio",
    "Occlusion",
]

# Regular range, excluding localized effects such as rashes or spots.
REGULAR_SKIN_MIN_VALUES = [0.005, 0.001, 0.005, 0.6, 0.72, 0.001]
REGULAR_SKIN_MAX_VALUES = [0.40, 0.0225, 0.025, 0.8, 0.76, 1.0]

# Plausible range.
SKIN_MIN_VALUES = [0.001, 0.001, 0.001, 0.001, 0.001, 0.001]
SKIN_MAX_VALUES = [0.45, 0.25, 0.035, 1.0, 1.0, 1.0]

# Full parameter range.
MIN_PROP_VALUES = [0.001, 0.001, 0.001, 0.001, 0.001, 0.001]
MAX_PROP_VALUES = [1.0, 1.0, 0.035, 1.0, 1.0, 1.0]

parameter_ranges = pd.DataFrame(
    {
        "Parameter": SKIN_PROPS,
        "Regular minimum": REGULAR_SKIN_MIN_VALUES,
        "Regular maximum": REGULAR_SKIN_MAX_VALUES,
        "Plausible minimum": SKIN_MIN_VALUES,
        "Plausible maximum": SKIN_MAX_VALUES,
        "Full minimum": MIN_PROP_VALUES,
        "Full maximum": MAX_PROP_VALUES,
    }
)

parameter_ranges

BioSkin does not distribute the original Monte Carlo human skin model used to generate training spectra. Instead, it provides a learned decoder that maps skin parameters directly to spectral reflectance. The demonstration below loads the pretrained decoder and evaluates one illustrative parameter vector.

In [17]:
from bioskin.bioskin import BioSkinInference
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 512000

model_path = "G:/Phd/BioSkin/pretrained_models/"
model_name = "BioSkinAO"

# Load the encoder-decoder model for skin reflectance reconstruction.
bio_skin = BioSkinInference(model_path + model_name, device=device, batch_size=batch_size)

# Define one illustrative input parameter vector.
skin_props = torch.tensor(
    [
        0.10,  # Melanin volume fraction
        0.01,  # Blood volume fraction
        0.01,  # Thickness
        0.70,  # Blood oxygenation
        0.74,  # Melanin type ratio
        0.10,  # Occlusion
    ],
    dtype=torch.float32,
    device=device,
).unsqueeze(0)

# Generate spectral reflectance with the pretrained decoder.
# Visible range: 380-780 nm, 2 nm resolution.
# Near-infrared range: 780-1000 nm, 2 nm resolution.
with torch.no_grad():
    ref_vis, ref_vis_rgb, ref_ir, ref_ir_avg = bio_skin.skin_props_to_reflectance(skin_props)

In [21]:
# Plot decoded reflectance in the visible and near-infrared ranges.
visible_wavelengths = 380 + 2 * torch.arange(ref_vis.shape[1], device=ref_vis.device)
infrared_wavelengths = 780 + 2 * torch.arange(ref_ir.shape[1], device=ref_ir.device)

visible_reflectance = ref_vis[0].detach().cpu().numpy()
infrared_reflectance = ref_ir[0].detach().cpu().numpy()
visible_wavelengths = visible_wavelengths.detach().cpu().numpy()
infrared_wavelengths = infrared_wavelengths.detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), dpi=150)
fig.suptitle("BioSkin decoded spectral reflectance", y=1.02, fontsize=13)

axes[0].plot(visible_wavelengths, visible_reflectance, color="#1f77b4", linewidth=2, label="Visible")
axes[0].plot(infrared_wavelengths, infrared_reflectance, color="#d62728", linewidth=2, label="Infrared")
axes[0].axvline(780, color="0.35", linestyle="--", linewidth=1)
axes[0].set_title("Full decoded spectrum")
axes[0].set_xlabel("Wavelength (nm)")
axes[0].set_ylabel("Reflectance (a.u.)")
axes[0].legend(frameon=False)

axes[1].plot(visible_wavelengths, visible_reflectance, color="#1f77b4", linewidth=2)
axes[1].set_title("Visible range")
axes[1].set_xlabel("Wavelength (nm)")

axes[2].plot(infrared_wavelengths, infrared_reflectance, color="#d62728", linewidth=2)
axes[2].set_title("Near-infrared range")
axes[2].set_xlabel("Wavelength (nm)")

for ax in axes:
    ax.grid(alpha=0.25)

plt.tight_layout()


### Interpretation

The figure reports the decoded reflectance for a single illustrative skin-parameter vector. The left panel shows the continuous decoded spectrum across the visible and near-infrared ranges, with the vertical line marking the 780 nm transition. The middle and right panels separate the two wavelength ranges to make local spectral structure easier to inspect.